# Calidad visual de las cámaras - Análisis Exploratorio de los Datos

El objetivo de este notebook es evaluar la utilidad de cada cámara para la investigación.

Se comprueba si la cámara permite identificar:
- peatones
- vehículos
- semáforos

Además, se mide la resolución, la luminosidad y la nitidez para detectar cámaras con baja calidad visual. Las imágenes se separan por **iluminación** (día / crepúsculo / noche, según la elevación solar): una imagen nocturna oscura no indica una cámara defectuosa.

Los criterios y umbrales viven en `config.yaml` (`calidad`) y el código en `src/calidad.py`, compartido con `02_preprocess_labelled_basic`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from calidad import add_quality, camera_blacklist, image_metrics, list_camera_images
from config import load_config, path
from tiempo import illumination, parse_timestamp_from_name

CFG = load_config()
RAW_DIR = path("raw")
print(f"Directorio de imágenes: {RAW_DIR}")

## Carga de imágenes

Se localizan las imágenes de `raw/<cámara>/` (se ignoran carpetas que no son cámaras, como `.ipynb_checkpoints`) y se calculan las métricas visuales. Para un análisis rápido puede limitarse el número de imágenes por cámara.

In [ ]:
MAX_POR_CAMARA = 200  # None = todas (más lento)

image_files = list_camera_images(RAW_DIR)
if MAX_POR_CAMARA is not None:
    by_camera = pd.Series(image_files).groupby([p.parent.name for p in image_files])
    image_files = [p for _, group in by_camera for p in group.sample(min(len(group), MAX_POR_CAMARA), random_state=42)]
print("Imágenes a analizar:", len(image_files))

lat, lon = CFG["ubicacion"]["lat"], CFG["ubicacion"]["lon"]
results = []
for img_path in image_files:
    try:
        metrics = image_metrics(img_path)
    except ValueError:
        continue
    ts = parse_timestamp_from_name(img_path.name)
    metrics["illumination"] = illumination(ts, lat, lon, CFG["iluminacion"]["elevacion_crepusculo"])
    results.append(metrics)

eda_visual = pd.DataFrame(results)
if eda_visual.empty:
    print("No hay imágenes válidas para analizar.")
else:
    display(eda_visual.head())
    print(eda_visual["illumination"].value_counts())

## Propiedades de las imágenes

Se analiza la resolución, iluminación y nitidez para discriminar cámaras útiles frente a cámaras con baja calidad visual. El `quality_score` combina los **percentiles** de brillo y nitidez (misma escala 0–1), en lugar de sus valores brutos, cuyas escalas difieren en un orden de magnitud.

In [ ]:
if not eda_visual.empty:
    print("Resolución:")
    print(eda_visual[["width", "height"]].value_counts().head(10))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.histplot(data=eda_visual, x="brightness", hue="illumination", bins=40, ax=axes[0])
    axes[0].set_title("Distribución del brillo por iluminación")
    sns.histplot(data=eda_visual, x="sharpness", bins=40, log_scale=True, ax=axes[1])
    axes[1].set_title("Distribución de la nitidez (escala log)")
    plt.tight_layout()
    plt.show()

    eda_visual, thresholds = add_quality(eda_visual, CFG["calidad"])
    print("Umbrales aplicados:", thresholds)
    print(f"Imágenes válidas: {eda_visual['valid'].mean():.1%}")

    print("Cámaras más oscuras de día:")
    print(eda_visual[eda_visual["illumination"] == "dia"].groupby("camera")["brightness"].mean().nsmallest(10))

    print("Cámaras más borrosas:")
    print(eda_visual.groupby("camera")["sharpness"].median().nsmallest(10))

    print("Mejores cámaras por score de utilidad:")
    print(eda_visual.groupby("camera")["quality_score"].mean().nlargest(10))
else:
    print("No hay datos suficientes para construir gráficos de calidad visual.")

## Cámaras candidatas a lista negra

Una cámara se descarta solo si falla más del `max_fraccion_fallos` de sus imágenes **de día**. La lista definitiva la guarda `02_preprocess_labelled_basic`.

In [ ]:
if not eda_visual.empty:
    stats = camera_blacklist(eda_visual, CFG["calidad"]["max_fraccion_fallos"])
    display(stats.sort_values("fraccion_fallos", ascending=False).head(15))
    print(f"Cámaras en lista negra: {stats['en_lista_negra'].sum()} de {len(stats)}")